# SART with phone call interruptions

Replication of the **call** and **no-notification** conditions of Stothart, Mitchum & Yehnert (2015). The SART is Stothart's original code (Python SART v2, MIT licence), with calls added. Calls come from an Android study phone over USB (ADB) instead of Twilio.

**Procedure (as in the paper)**
- Participants are randomly assigned to `call` or `no_call` by the notebook, so the experimenter doesn't know the condition in advance. The observer may still notice the calls during block 2.
- 18 practice trials with feedback, then 2 blocks of 360 trials, with a 60 s break in between.
- Digit 250 ms, mask 900 ms. Press space for every digit except 3.
- Call condition: calls to the participant's own phone during block 2, right after trials 1, 91, 181 and 271.
- Participants get no instructions about their phone. Everything else is identical for both conditions.

**Additions**
- Each call is a short pre-call followed by the real call, so it gets through Do Not Disturb (repeated-caller exception).
- A call is hung up as soon as it is answered (by the participant or voicemail).

**Setup:** this notebook runs on the participant's computer, in a Python environment with PsychoPy. The study phone is plugged in by USB, hidden and on silent.

**Before each session:** run all cells up to section 7, then section 7 starts the task.

## 1. Settings

In [64]:
import csv
import os
import random
import re
import subprocess
import sys
import threading
import time
from datetime import datetime

import pandas as pd
from psychopy import visual, core, data, event, gui

# --- Phone ---
SIMULATE = True       # True: no real calls, for dry runs
# Full path to adb.exe on this computer. Change it to where the platform-tools
# folder is.
ADB_PATH = r"C:\Users\xxxxxxxx\platform-tools\adb.exe"
ADB_SERIAL = None      # only needed if more than one phone is connected
ADB_TIMEOUT = 10       # seconds

# --- Calls ---
CALL_BLOCK = 2
CALL_AFTER_TRIALS = [1, 91, 181, 271]
CALL_DURATION = 17         # max length of the real call (s), from dialling to hang-up. Not given
                           # in the paper; must stay below the voicemail delay (about 21 s in our test)
PRECALL_RING_TIME = 1.0    # pre-call is hung up this long after it starts ringing (s)
PRECALL_TIMEOUT = 15       # ...or after this long if it never rings (s)
GAP_BETWEEN_CALLS = 1.0    # pause between pre-call and real call (s)
CALL_CHECK_INTERVAL = 0.5  # how often the call state is checked (s)

# --- SART (as in the paper) ---
# Screen profile. "testMonitor" is PsychoPy's default: the task works, but the
# digit sizes (in cm) may not match the paper. For exact sizes, make a profile
# for the screen you use and put its name here. Keep it the same for all participants.
MONITOR = "testMonitor"
BLOCKS = 2
REPS = 8                   # 8 x 45 = 360 trials per block
OMIT_NUM = 3
PRACTICE = True
BREAK_TIME = 60            # seconds

# --- Files ---
PARTICIPANTS_FILE = "participants.csv"   # participant,phone (keep private)
CONDITIONS_FILE = "conditions.csv"       # condition per participant, filled in by the notebook
DATA_PATH = ""                           # folder for output files ("" = this folder)

## 2. Participant phone numbers

`participants.csv` has two columns, `participant` and `phone` (for example `12,+31612345678`). The participant number is the one typed in the start-up dialog. Phone numbers never go into the data files.

In [65]:
def get_phone(participant):

    # Read as text so the + in phone numbers is kept
    table = pd.read_csv(PARTICIPANTS_FILE, dtype=str)
    numbers = dict(zip(table["participant"].str.strip(), table["phone"].str.strip()))

    participant = str(participant).strip()
    if participant not in numbers:
        raise ValueError(f"Participant '{participant}' is not in {PARTICIPANTS_FILE}.")

    phone = numbers[participant]
    if not re.fullmatch(r"\+\d{8,15}", phone):
        raise ValueError(f"Phone number of '{participant}' should look like +31612345678.")

    return phone

## 2b. Condition assignment

Random, in groups of four (two `call`, two `no_call`, shuffled), so both conditions end up about the same size. The result is stored in `conditions.csv` and never printed. Participant `TEST` always gets `call` and is not stored.

In [66]:
def assign_condition(participant):

    if participant == "TEST":
        return "call"

    if os.path.exists(CONDITIONS_FILE):
        table = pd.read_csv(CONDITIONS_FILE, dtype=str)
    else:
        table = pd.DataFrame(columns=["participant", "condition", "datetime"])

    # Someone already assigned keeps their condition
    existing = table.loc[table["participant"] == participant, "condition"]
    if len(existing):
        return existing.iloc[0]

    # Conditions still open in the current group of four
    open_slots = ["call", "call", "no_call", "no_call"]
    for used in table["condition"].iloc[len(table) - len(table) % 4:]:
        open_slots.remove(used)
    condition = random.choice(open_slots)

    new_row = pd.DataFrame([{"participant": participant, "condition": condition,
                             "datetime": datetime.now().isoformat(timespec="seconds")}])
    pd.concat([table, new_row]).to_csv(CONDITIONS_FILE, index=False)

    return condition

## 3. Phone control (ADB)

The study phone dials with Android's own dialer and hangs up with a key press. Android also reports the call state: dialling, ringing (the participant's phone rings) or answered.

In [67]:
IDLE, ANSWERED, DIALING, RINGING = 0, 1, 3, 4   # Android's call state numbers

simulated = threading.local()   # pretend call state, used when SIMULATE is True


def adb(*arguments):

    command = [ADB_PATH] + (["-s", ADB_SERIAL] if ADB_SERIAL else []) + ["shell", *arguments]
    result = subprocess.run(command, capture_output=True, text=True, timeout=ADB_TIMEOUT)

    if result.returncode != 0:
        raise RuntimeError("ADB failed: " + result.stderr.strip())

    return result.stdout + result.stderr


def check_phone():

    # Checks that exactly one phone (or the chosen one) is ready
    result = subprocess.run([ADB_PATH, "devices"], capture_output=True, text=True, timeout=ADB_TIMEOUT)
    ready = [line.split("\t")[0] for line in result.stdout.splitlines()[1:] if line.endswith("\tdevice")]

    if ADB_SERIAL and ADB_SERIAL not in ready:
        raise RuntimeError(f"Phone {ADB_SERIAL} is not connected.")
    if not ADB_SERIAL and len(ready) != 1:
        raise RuntimeError(f"Expected 1 connected phone, found {len(ready)}. Check cable and USB debugging.")

    call_state()   # fails if this phone does not report call states
    print("Study phone ready:", ready[0] if not ADB_SERIAL else ADB_SERIAL)


def dial(phone):

    if SIMULATE:
        simulated.start = time.perf_counter()
        return

    output = adb("am", "start", "-a", "android.intent.action.CALL", "-d", f"tel:{phone}")
    if "Error" in output:
        raise RuntimeError(output.strip())


def hang_up():

    if SIMULATE:
        simulated.start = None
        return

    adb("input", "keyevent", "KEYCODE_ENDCALL")


def call_state():

    if SIMULATE:
        start = getattr(simulated, "start", None)
        if start is None:
            return IDLE
        return DIALING if time.perf_counter() - start < 0.5 else RINGING

    found = [int(n) for n in re.findall(r"Foreground call state: ?(-?\d+)", adb("dumpsys", "telephony.registry"))]
    if not found:
        raise RuntimeError("The phone does not report its call state.")

    # Phones with two SIM slots report more than one; take the busiest
    for state in (ANSWERED, RINGING, DIALING):
        if state in found:
            return state
    return IDLE


def wait_until_idle(timeout=5):

    end = time.perf_counter() + timeout
    while time.perf_counter() < end:
        try:
            if call_state() == IDLE:
                return
        except Exception:
            pass
        time.sleep(CALL_CHECK_INTERVAL)

## 4. Calls

`make_call` runs in the background so the SART timing is not affected. It makes a pre-call and then the real call, and logs every step:

| Event | Meaning |
|---|---|
| `_START` | study phone starts dialling |
| `_RINGING` | participant's phone is ringing |
| `_ANSWERED` | picked up (participant or voicemail), hung up straight away |
| `_END` | hung up by the notebook |
| `_ENDED` | stopped on the other side (declined, busy, unreachable) |

Each event starts with `PRE` or `MAIN`. A failed command is logged as `..._FAILED`. Times use `time.perf_counter()`, the same clock as the SART.

In [68]:
call_log = []       # filled during the session
call_threads = []


def log_event(participant, trial, event, error=""):

    call_log.append({
        "participant": participant,
        "block": CALL_BLOCK,
        "trial": trial,
        "event": event,
        "time_s": time.perf_counter(),
        "datetime": datetime.now().isoformat(timespec="milliseconds"),
        "error": error,
    })


def one_call(participant, trial, part, phone, max_time, stop_after_ringing=None):

    def end_call():
        # Try a few times, so a short USB hiccup can't leave a call running
        for attempt in range(5):
            try:
                hang_up()
                return log_event(participant, trial, f"{part}_END")
            except Exception as problem:
                error = str(problem)
                time.sleep(1)
        log_event(participant, trial, f"{part}_END_FAILED", error)

    try:
        dial(phone)
        log_event(participant, trial, f"{part}_START")
    except Exception as problem:
        log_event(participant, trial, f"{part}_START_FAILED", str(problem))
        return

    start = time.perf_counter()
    call_seen = False
    ringing_since = None

    while True:
        time.sleep(CALL_CHECK_INTERVAL)
        now = time.perf_counter()

        try:
            state = call_state()
        except Exception:
            state = None   # could not read this time; keep going

        if state in (DIALING, RINGING, ANSWERED):
            call_seen = True

        if state == RINGING and ringing_since is None:
            ringing_since = now
            log_event(participant, trial, f"{part}_RINGING")

        if state == ANSWERED:
            log_event(participant, trial, f"{part}_ANSWERED")
            return end_call()

        if state == IDLE and (call_seen or now - start > 5):
            return log_event(participant, trial, f"{part}_ENDED")

        if stop_after_ringing and ringing_since and now - ringing_since >= stop_after_ringing:
            return end_call()

        if now - start >= max_time:
            return end_call()


def make_call(participant, trial, duration=CALL_DURATION):

    phone = get_phone(participant)

    def sequence():
        # Short pre-call, so the real call gets through Do Not Disturb
        one_call(participant, trial, "PRE", phone, PRECALL_TIMEOUT, stop_after_ringing=PRECALL_RING_TIME)
        wait_until_idle()
        time.sleep(GAP_BETWEEN_CALLS)
        one_call(participant, trial, "MAIN", phone, duration)
        wait_until_idle()

    thread = threading.Thread(target=sequence, daemon=True)
    thread.start()
    call_threads.append(thread)


def wait_for_calls():

    for thread in call_threads:
        thread.join()
    call_threads.clear()


def ringing_spans():

    # Time spans in which the participant's phone was ringing
    spans = []
    for ring in [e for e in call_log if e["event"].endswith("_RINGING")]:
        part = ring["event"].split("_")[0]
        stops = [e["time_s"] for e in call_log
                 if e["trial"] == ring["trial"] and e["time_s"] >= ring["time_s"]
                 and e["event"] in (f"{part}_ANSWERED", f"{part}_END", f"{part}_ENDED")]
        if stops:
            spans.append((ring["time_s"], min(stops)))
    return spans

## 5. Phone test

Before the first session: add a `TEST` row with your own number to `participants.csv`, set `SIMULATE = False`, and run this cell. Your phone should ring briefly, stop, and ring again. Run it once more and answer the second call: it should hang up straight away. The log should show `RINGING`, and `ANSWERED` in the second test.

In [69]:
# check_phone()
# call_log.clear()
# make_call("TEST", trial=0, duration=15)
# wait_for_calls()
# pd.DataFrame(call_log)

## 6. SART

Stothart's original code, unchanged except for:
- `sart_block` calls `on_trial_end` after each trial (used to start calls);
- the break length comes from `BREAK_TIME`;
- `sart` checks the participant and phone before the task starts, assigns the condition, starts the calls (call condition only), adds three columns to the data file (`condition`, `call_after_trial`, `phone_ringing`), saves the call log, and closes the window at the end.

In [70]:
def sart(monitor=MONITOR, blocks=BLOCKS, reps=REPS, omitNum=OMIT_NUM,
         practice=PRACTICE, path=DATA_PATH, fixed=False):

    partInfo = part_info_gui()
    participant = str(partInfo[0]).strip()
    fileName = "SART_" + participant + ".txt"
    callFileName = "calls_" + participant + ".csv"

    # --- Checks before the task starts ---
    if os.path.exists(path + fileName):
        raise FileExistsError(fileName + " already exists. Use another participant number.")
    get_phone(participant)
    if not SIMULATE:
        check_phone()   # checked for everyone, so it doesn't reveal the condition

    # Chosen by the notebook; not shown to the experimenter
    condition = assign_condition(participant)

    call_log.clear()
    callTrials = set()

    def on_trial_end(bNum, tNum):
        # Start a call right after the scheduled trials of the call block
        if condition == "call" and bNum == CALL_BLOCK and tNum in CALL_AFTER_TRIALS:
            callTrials.add((bNum, tNum))
            make_call(participant, tNum)

    mainResultList = []
    win = visual.Window(fullscr=True, color="black", units='cm', monitor=monitor)

    try:
        sart_init_inst(win, omitNum)
        if practice == True:
            sart_prac_inst(win, omitNum)
            mainResultList.extend(sart_block(win, fb=True, omitNum=omitNum,
                                  reps=1, bNum=0, fixed=fixed))
        sart_act_task_inst(win)
        for block in range(1, blocks + 1):
            mainResultList.extend(sart_block(win, fb=False, omitNum=omitNum,
                                  reps=reps, bNum=block, fixed=fixed,
                                  on_trial_end=on_trial_end))
            if (blocks > 1) and (block != blocks):
                sart_break_inst(win)

    finally:
        # Let running calls finish (a call is always hung up), then save
        wait_for_calls()
        win.close()

        spans = ringing_spans()
        with open(path + fileName, "w") as outFile:
            outFile.write("part_num\tpart_gender\tpart_age\tpart_school_yr\t" +
                          "part_normal_vision\texp_initials\tblock_num\ttrial_num\t" +
                          "number\tomit_num\tresp_acc\tresp_rt\ttrial_start_time_s" +
                          "\ttrial_end_time_s\tmean_trial_time_s\tcondition\tcall_after_trial" +
                          "\tphone_ringing\ttiming_function\n")
            for line in mainResultList:
                start = float(line[6])
                callAfter = (int(line[0]), int(line[1])) in callTrials
                ringing = any(a <= start < b for a, b in spans)
                for item in partInfo:
                    outFile.write(str(item) + "\t")
                for col in line:
                    outFile.write(str(col) + "\t")
                outFile.write(condition + "\t" + str(int(callAfter)) + "\t" + str(int(ringing)) + "\t")
                outFile.write("time.perf_counter()\n")

        # Saved for both conditions (empty for no_call), so the files don't reveal it
        pd.DataFrame(call_log, columns=["participant", "block", "trial", "event",
                                        "time_s", "datetime", "error"]).to_csv(path + callFileName, index=False)
        print("Saved", fileName, "and", callFileName)


def part_info_gui():
    info = gui.Dlg(title='SART')
    info.addText('Participant Info')
    info.addField('Part. Number: ')
    info.addField('Part. Gender: ',
                  choices=["Please Select", "Male", "Female", "Other"])
    info.addField('Part. Age:  ')
    info.addField('Part. Year in School: ',
                  choices=["Please Select", "Freshman", "Sophmore", "Junior",
                           "Senior", "1st Year Graduate Student",
                           "2nd Year Graduate Student",
                           "3rd Year Graduate Student",
                           "4th Year Graduate Student",
                           "5th Year Graduate Student",
                           "6th Year Graduate Student"])
    info.addField('Do you have normal or corrected-to-normal vision?',
                  choices=["Please Select", "Yes", "No"])
    info.addText('Experimenter Info')
    info.addField('DIS Initials:  ')
    info.show()
    if info.OK:
        infoData = info.data
    else:
        sys.exit()
    return infoData


def sart_init_inst(win, omitNum):
    inst = visual.TextStim(win, text=("In this task, a series of numbers will" +
                                      " be presented to you.  For every" +
                                      " number that appears except for the" +
                                      " number " + str(omitNum) + ", you are" +
                                      " to press the space bar as quickly as" +
                                      " you can.  That is, if you see any" +
                                      " number but the number " +
                                      str(omitNum) + ", press the space" +
                                      " bar.  If you see the number " +
                                      str(omitNum) + ", do not press the" +
                                      " space bar or any other key.\n\n" +
                                      "Please give equal importance to both" +
                                      " accuracy and speed while doing this" +
                                      " task.\n\nPress the b key when you" +
                                      " are ready to start."),
                           color="white", height=0.7, pos=(0, 0))
    event.clearEvents()
    while 'b' not in event.getKeys():
        inst.draw()
        win.flip()


def sart_prac_inst(win, omitNum):
    inst = visual.TextStim(win, text=("We will now do some practice trials " +
                                      "to familiarize you with the task.\n" +
                                      "\nRemember, press the space bar when" +
                                      " you see any number except for the " +
                                      " number " + str(omitNum) + ".\n\n" +
                                      "Press the b key to start the " +
                                      "practice."),
                           color="white", height=0.7, pos=(0, 0))
    event.clearEvents()
    while 'b' not in event.getKeys():
        inst.draw()
        win.flip()


def sart_act_task_inst(win):
    inst = visual.TextStim(win, text=("We will now start the actual task.\n" +
                                      "\nRemember, give equal importance to" +
                                      " both accuracy and speed while doing" +
                                      " this task.\n\nPress the b key to " +
                                      "start the actual task."),
                           color="white", height=0.7, pos=(0, 0))
    event.clearEvents()
    while 'b' not in event.getKeys():
        inst.draw()
        win.flip()


def sart_break_inst(win):
    inst = visual.TextStim(win, text=("You will now have a " + str(BREAK_TIME) +
                                      " second break.  Please remain in your " +
                                      "seat during the break."),
                           color="white", height=0.7, pos=(0, 0))
    nbInst = visual.TextStim(win, text=("You will now do a new block of" +
                                        " trials.\n\nPress the b key " +
                                        "bar to begin."),
                             color="white", height=0.7, pos=(0, 0))
    startTime = time.perf_counter()
    while 1:
        eTime = time.perf_counter() - startTime
        inst.draw()
        win.flip()
        if eTime > BREAK_TIME:
            break
    event.clearEvents()
    while 'b' not in event.getKeys():
        nbInst.draw()
        win.flip()


def sart_block(win, fb, omitNum, reps, bNum, fixed, on_trial_end=None):
    mouse = event.Mouse(visible=0)
    xStim = visual.TextStim(win, text="X", height=3.35, color="white",
                            pos=(0, 0))
    circleStim = visual.Circle(win, radius=1.50, lineWidth=8,
                               lineColor="white", fillColor=None, pos=(0, .1))
    numStim = visual.TextStim(win, font="Arial", color="white", pos=(0, 0))
    correctStim = visual.TextStim(win, text="CORRECT", color="green",
                                  font="Arial", pos=(0, 0))
    incorrectStim = visual.TextStim(win, text="INCORRECT", color="red",
                                    font="Arial", pos=(0, 0))
    numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9]
    if fb == True:
        fontSizes = [1.20, 3.00]
    else:
        fontSizes = [1.20, 1.80, 2.35, 2.50, 3.00]
    list = data.createFactorialTrialList({"number": numbers,
                                          "fontSize": fontSizes})
    seqList = []
    for i in range(len(fontSizes)):
        for number in numbers:
            random.shuffle(list)
            for trial in list:
                if trial["number"] == number and trial not in seqList:
                    seqList.append(trial)
                    break
    if fixed == True:
        trials = data.TrialHandler(seqList, nReps=reps, method='sequential')
    else:
        trials = data.TrialHandler(list, nReps=reps, method='random')
    clock = core.Clock()
    tNum = 0
    resultList = []
    startTime = time.perf_counter()
    for trial in trials:
        tNum += 1
        resultList.append(sart_trial(win, fb, omitNum, xStim, circleStim,
                                     numStim, correctStim, incorrectStim, clock,
                                     trials.thisTrial['fontSize'],
                                     trials.thisTrial['number'], tNum, bNum, mouse))
        if on_trial_end is not None:
            on_trial_end(bNum, tNum)
    endTime = time.perf_counter()
    totalTime = endTime - startTime
    for row in resultList:
        row.append(totalTime/tNum)
    print("\n\n#### Block " + str(bNum) + " ####\nDes. Time Per P Trial: " +
          str(2.05*1000) + " ms\nDes. Time Per Non-P Trial: " +
          str(1.15*1000) + " ms\nActual Time Per Trial: " +
          str((totalTime/tNum)*1000) + " ms\n\n")
    return resultList


def sart_trial(win, fb, omitNum, xStim, circleStim, numStim, correctStim,
               incorrectStim, clock, fontSize, number, tNum, bNum, mouse):
    startTime = time.perf_counter()
    mouse.setVisible(0)
    respRt = "NA"
    numStim.setHeight(fontSize)
    numStim.setText(number)
    numStim.draw()
    event.clearEvents()
    clock.reset()
    stimStartTime = time.perf_counter()
    win.flip()
    xStim.draw()
    circleStim.draw()
    waitTime = .25 - (time.perf_counter() - stimStartTime)
    core.wait(waitTime, hogCPUperiod=waitTime)
    maskStartTime = time.perf_counter()
    win.flip()
    waitTime = .90 - (time.perf_counter() - maskStartTime)
    core.wait(waitTime, hogCPUperiod=waitTime)
    win.flip()
    allKeys = event.getKeys(timeStamped=clock)
    if len(allKeys) != 0:
        respRt = allKeys[0][1]
    if len(allKeys) == 0:
        if omitNum == number:
            respAcc = 1
        else:
            respAcc = 0
    else:
        if omitNum == number:
            respAcc = 0
        else:
            respAcc = 1
    if fb == True:
        if respAcc == 0:
            incorrectStim.draw()
        else:
            correctStim.draw()
        stimStartTime = time.perf_counter()
        win.flip()
        waitTime = .90 - (time.perf_counter() - stimStartTime)
        core.wait(waitTime, hogCPUperiod=waitTime)
        win.flip()
    endTime = time.perf_counter()
    totalTime = endTime - startTime
    return [str(bNum), str(tNum), str(number), str(omitNum), str(respAcc),
            str(respRt), str(startTime), str(endTime)]

## 7. Run the session

Opens the participant dialog, then the task full screen. Saves `SART_<number>.txt` (trial data) and `calls_<number>.csv` (call log). The printed "Actual Time Per Trial" after each block should be close to 1150 ms.

In [ ]:
sart()

## 8. Experimenter notes

Fill in after debriefing. The paper excluded participants who looked at or handled their phone, or whose phone was off, and asked for the alert setting. Also note whether the phone allows repeated callers through Do Not Disturb.

In [71]:
def save_notes(participant, times_looked_or_handled, phone_off, alert_setting, repeat_callers_allowed="", other=""):

    row = pd.DataFrame([{
        "participant": participant,
        "times_looked_or_handled": times_looked_or_handled,
        "phone_off": phone_off,
        "alert_setting": alert_setting,                 # sound / vibrate / silent / do not disturb
        "repeat_callers_allowed": repeat_callers_allowed,
        "other": other,
        "datetime": datetime.now().isoformat(timespec="seconds"),
    }])
    notes_file = DATA_PATH + "session_notes.csv"
    row.to_csv(notes_file, mode="a", header=not os.path.exists(notes_file), index=False)
    print("Notes saved.")


# save_notes("12", times_looked_or_handled=0, phone_off=False, alert_setting="vibrate", repeat_callers_allowed="yes")